# 🔍 Pipeline End-to-End de Retenção de Clientes: SQL Server para TensorFlow

## 📌 Contexto de Negócio
No varejo de alta escala, prever a evasão de clientes (*Churn*) é uma das estratégias mais críticas para manter a saúde financeira e otimizar o custo de aquisição de clientes (CAC). Este projeto simula um caso real corporativo utilizando a base de dados **Contoso Retail DW**, contendo **18.484 registros de clientes**.

O principal desafio deste problema foi lidar com o **desbalanceamento severo de dados**: apenas 10,9% da base histórica correspondia a clientes inativos (*churn*). Um modelo ingênuo apresentaria uma acurácia falsa de ~89%, mas seria completamente incapaz de detectar o risco real de evasão.

---

## 🛠️ O que eu aprendi e apliquei neste projeto (Competências Adquiridas)

1. **Engenharia de Dados com SQLAlchemy (ORM):** Abandonei scripts com cursores manuais (`pyodbc` puro) para utilizar uma arquitetura ORM padrão de mercado. Isso garante conexões seguras, reutilizáveis e fáceis de portar para ambientes de nuvem ou outros bancos relacionais.
2. **Arquitetura de Deep Learning (TensorFlow/Keras):** Construí uma Rede Neural multicamadas do zero, compreendendo o fluxo estatístico de ajuste de pesos através de funções de perda (`binary_crossentropy`) e otimizadores avançados (`adam`).
3. **Mecanismos de Prevenção de Overfitting (Callbacks):** Implementei componentes de automação de treinamento como o `EarlyStopping` e o `ReduceLROnPlateau`. Entendi na prática como interromper o treinamento antes que a rede decore os dados de treino e perca o poder de generalização na validação.
4. **Análise Crítica de Métricas Logísticas/Comerciais:** Compreendi que no problema de Churn, a métrica de **Recall** é soberana à Acurácia. É preferível que o sistema gere pequenos alarmes falsos controlados do que deixar um cliente de alto valor evadir sem aviso.

---

## 🏗️ Estrutura e Fluxo do Projeto (End-to-End)

```text
  [ SQL Server ]  ---> [ SQLAlchemy ORM ] ---> [ One-Hot Encoding ]
(Contoso Retail DW)       (Extração Limpa)        (Categorias -> Números)
                                                          |
  [ Avaliação Final ] <--- [ EarlyStopping ] <--- [ StandardScaler ]
  (Recall Otimizado)       (Treino Seguro)      (Normalização de Dados)

In [1]:
# ==============================================================================
# BLOCO 1: AMBIENTE E DEPENDÊNCIAS
# Objetivo: Importar os módulos necessários para conexão ORM, processamento de dados,
#           construção de redes neurais e avaliação de métricas.
# ==============================================================================

import os
import pyodbc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Conexão e ORM Banco de Dados
import sqlalchemy as sa
from sqlalchemy import create_engine

# Pré-processamento e Validação
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Inteligência Artificial / Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Algoritmos Comparativos Baseline
from sklearn.ensemble import RandomForestClassifier

In [2]:
# ==============================================================================
# BLOCO 2: ENGENHARIA DE DADOS & ORM
# Objetivo: Estabelecer conexão profissional com o Contoso Retail DW usando
#           SQLAlchemy, eliminando cursores manuais e permitindo consultas escaláveis.
# ==============================================================================

# String de conexão utilizando o driver ODBC nativo do SQL Server
SERVER_NAME = "ARAUJO"  # Alterar de acordo com o ambiente local
DATABASE_NAME = "ContosoRetailDW"

connection_string = f"Driver={{ODBC Driver 17 for SQL Server}};Server={SERVER_NAME};Database={DATABASE_NAME};Trusted_Connection=yes;"
connection_url = sa.engine.URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

# Criação do motor (engine) de conexão do SQLAlchemy
engine = create_engine(connection_url)
print("Conexão ORM estabelecida com sucesso com o SQL Server!")

Conexão ORM estabelecida com sucesso com o SQL Server!


In [3]:
# ==============================================================================
# BLOCO 3: EXTRAÇÃO & CONSULTA REGRAS DE NEGÓCIO
# Objetivo: Trazer a base histórica de clientes (18.484 registros) mapeando
#           as features demográficas, frequência e a variável alvo (Churn).
# ==============================================================================

query = """
 SELECT 
        CustomerKey,
        Idade,
        TotalItensComprados,
        TotalItensDevolvidos,
        TotalPedidos, 
        ValorTotalGasto, 
        TicketMedio,
        DeixouDeComprar
    FROM dbo.vw_Dados_Treinamento_Churn
"""

# Leitura direta do banco para o DataFrame do Pandas
df = pd.read_sql_query(query, engine)

# Diagnóstico de desbalanceamento crítico da classe alvo
print(df['DeixouDeComprar'].value_counts(normalize=True))

DeixouDeComprar
0    0.891257
1    0.108743
Name: proportion, dtype: float64


In [4]:
df.head()

,CustomerKey,Idade,TotalItensComprados,TotalItensDevolvidos,TotalPedidos,ValorTotalGasto,TicketMedio,DeixouDeComprar
0,6498,46,113,1,57,35857.3745,314.5383,0
1,2253,68,214,0,122,69182.5488,323.2829,0
2,14988,64,106,0,55,36276.7310,342.2333,0
3,10743,61,89,0,56,29023.6955,326.1089,0
4,7185,79,214,1,119,55134.8343,256.4410,0


In [5]:
# ==============================================================================
# BLOCO 4: PRÉ-PROCESSAMENTO & ENCODING
# Objetivo: Tratar variáveis categóricas (texto) transformando-as em numéricas
#           através de One-Hot Encoding, permitindo que a Rede Neural processe os dados.
# ==============================================================================

# Elimina identificadores que não geram padrão estatístico
df_ml = df.drop(columns=['CustomerKey'])

# Aplica One-Hot Encoding para colunas categóricas (Gênero, Estado Civil, Escolaridade)
df_ml = pd.get_dummies(df_ml, drop_first=True)

# Divisão de Features (X) e Alvo (y)
X = df_ml.drop(columns=['DeixouDeComprar'])
y = df_ml['DeixouDeComprar']

In [6]:
# ==============================================================================
# BLOCO 5: SEPARAÇÃO DE DADOS & ESCALONAMENTO
# Objetivo: Dividir a base em Treino, Validação e Teste (70/15/15) e aplicar 
#           StandardScaler para evitar distorção de pesos na Rede Neural.
# ==============================================================================

# Divisão inicial: Treino e Resto
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Divisão do resto: Validação e Teste em partes iguais
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Escalonamento obrigatório para Deep Learning (Média=0, Desvio Padrão=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [7]:
# ==============================================================================
# BLOCO 6: COMPENSAÇÃO DE DESBALANCEAMENTO
# Objetivo: Calcular pesos para equilibrar o aprendizado da IA, forçando a
#           Rede Neural a penalizar severamente os erros na classe minoritária (Churn).
# ==============================================================================

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, weights))

print(f"Pesos calculados automaticamente para equilibrar o negócio: {class_weights_dict}")

Pesos calculados automaticamente para equilibrar o negócio: {np.int64(0): np.float64(0.561009452779464), np.int64(1): np.float64(4.59772565742715)}


In [8]:
# ==============================================================================
# BLOCO 7: CONSTRUÇÃO DA ARQUITETURA DE DEEP LEARNING
# Objetivo: Definir uma rede neural multicamadas (Dense) com camadas de Dropout
#           para controle de regularização e prevenção de overfitting.
# ==============================================================================

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3), # Desativa 30% dos neurônios para forçar o aprendizado generalista
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Ativação final Sigmoide gera probabilidade entre 0 e 1
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

C:\Users\positivo\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# ==============================================================================
# BLOCO 8: MECANISMOS DE SEGURANÇA E TREINAMENTO
# Objetivo: Adicionar Callbacks estratégicos (EarlyStopping) para interromper o 
#           treinamento caso a performance de validação pare de melhorar.
# ==============================================================================

callbacks_list = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001, verbose=1)
]

# Execução do treinamento da Rede Neural
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=callbacks_list,
    verbose=1
)

Epoch 1/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8635 - loss: 0.3674 - recall: 0.8244 - val_accuracy: 0.9780 - val_loss: 0.2151 - val_recall: 0.8775 - learning_rate: 0.0010
Epoch 2/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9717 - loss: 0.2188 - recall: 0.8529 - val_accuracy: 0.9820 - val_loss: 0.1247 - val_recall: 0.8775 - learning_rate: 0.0010
Epoch 3/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9774 - loss: 0.1991 - recall: 0.8564 - val_accuracy: 0.9856 - val_loss: 0.1138 - val_recall: 0.8775 - learning_rate: 0.0010
Epoch 4/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9809 - loss: 0.1863 - recall: 0.8586 - val_accuracy: 0.9859 - val_loss: 0.1140 - val_recall: 0.8775 - learning_rate: 0.0010
Epoch 5/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9821 - loss: 0.1791 - recall: 0.8593 - val_accuracy: 0.9859 - val_loss: 0.1069 - val_recall: 0.8775 - learning_rate: 0.0010
Epoch 6/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/ste

In [10]:
# ==============================================================================
# BLOCO 9: VALIDAÇÃO TÉCNICA E ANÁLISE DE CUSTO
# Objetivo: Avaliar o modelo final na base de testes isolada, gerando a
#           matriz de confusão e avaliando a acurácia real e o Recall obtidos.
# ==============================================================================

y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("--- RELATÓRIO DE CLASSIFICAÇÃO DA REDE NEURAL ---")
print(classification_report(y_test, y_pred))

# Cálculo da Área sob a Curva ROC (Poder de discriminação)
auc_score = roc_auc_score(y_test, y_pred_prob)
print(f"ROC-AUC Score Final: {auc_score:.4f}")

87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
--- RELATÓRIO DE CLASSIFICAÇÃO DA REDE NEURAL ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2472
           1       0.99      0.88      0.93       301

    accuracy                           0.99      2773
   macro avg       0.99      0.94      0.96      2773
weighted avg       0.99      0.99      0.99      2773

ROC-AUC Score Final: 0.9717
